# Fine-tuning một mô hình reranker `BAAI/bge-reranker-v2-m3` cho truy xuất văn bản pháp luật tiếng Việt

## 0. Mục tiêu bài toán

Bài toán được mô hình hóa như một bài toán **reranking** trong hệ thống truy xuất văn bản pháp luật tiếng Việt. Với một câu hỏi pháp luật `query` và một danh sách các đoạn văn bản ứng viên `passages`, mô hình cần xếp đoạn đúng hoặc liên quan nhất lên vị trí cao nhất.

Trong pipeline RAG, reranker thường được đặt sau bước truy xuất ban đầu:

```text
User query
    ↓
Retriever / BM25 / FAISS lấy top-k documents
    ↓
Reranker chấm điểm từng cặp (query, passage)
    ↓
Sắp xếp lại candidates theo điểm reranker
    ↓
Đưa top passages vào LLM hoặc trả kết quả cho người dùng
```

Mục tiêu chính:

1. Fine-tune `BAAI/bge-reranker-v2-m3` trên dữ liệu legal QA tiếng Việt.
2. So sánh với các baseline/phương pháp khác.
3. Xuất đầy đủ số liệu, bảng biểu và hình ảnh phục vụ báo cáo.


## 1. Cài đặt thư viện


In [1]:
!pip install -U torch transformers datasets accelerate scikit-learn tqdm pandas sentencepiece protobuf matplotlib

In [2]:
# Fix lỗi: RuntimeError: operator torchvision::nms does not exist
# Nguyên nhân thường là torchvision không tương thích với torch trong runtime.
# Với bài toán NLP embedding/reranking, ta không cần torchvision.
import os
import sys
import subprocess
import gc
import json
import math
import random
import inspect
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from datasets import load_dataset
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    check=False,
)

print("Đã gỡ torchvision để tránh lỗi import transformers.")


ModuleNotFoundError: Could not import module 'Trainer'. Are this object's requirements defined correctly?

## 2. Mount Google Drive và thiết lập thư mục làm việc


In [ ]:
import os

from google.colab import drive; drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive"
PROJECT_DIR = "/content/drive/MyDrive/Project_ML"

if not os.path.exists(DRIVE_DIR):
    raise RuntimeError(
        "from google.colab import drive; drive.mount('/content/drive')"
    )

os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project dir:", PROJECT_DIR)

## 3. Import thư viện và cấu hình thí nghiệm

Tỷ lệ chia dữ liệu được đặt là **70/15/15** cho Train/Validation/Test.

Lý do lựa chọn:

- **Train 70%**: đủ lớn để fine-tune mô hình.
- **Validation 15%**: dùng để theo dõi loss/metric theo epoch và chọn checkpoint tốt.
- **Test 15%**: chỉ dùng cho đánh giá cuối cùng, tránh dùng test để điều chỉnh mô hình.

Vì reranker tiêu tốn VRAM theo số cặp `(query, passage)`, batch size được tính theo **số query-group**, không phải số cặp văn bản.

In [ ]:
# ===== Cấu hình chính =====
MODEL_NAME = "BAAI/bge-reranker-v2-m3"
OUTPUT_PATH = "./vietlaw-bge-reranker-v2-m3-finetuned"

SEED = 42

# Chia dữ liệu theo yêu cầu báo cáo: Train / Validation / Test.
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-9

# BGE reranker v2-m3 hỗ trợ ngữ cảnh dài, nhưng training dài rất tốn VRAM.
# Nếu GPU mạnh, có thể tăng 1024 -> 2048.
MAX_LENGTH = 1024

# Mỗi query trong train có 1 positive + TRAIN_NEGATIVES_PER_QUERY negatives.
TRAIN_NEGATIVES_PER_QUERY = 3

# Batch size tính theo số query-group, không phải số cặp.
# Số forward pairs thực tế = PER_DEVICE_TRAIN_BATCH_SIZE * (1 + TRAIN_NEGATIVES_PER_QUERY)
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

EPOCHS = 2
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01

# Sinh negative: "tfidf" tạo semi-hard negatives; "random" nhanh hơn nhưng dễ hơn.
NEGATIVE_STRATEGY = "tfidf"   # "tfidf" hoặc "random"
TFIDF_TOP_K_POOL = 50

# Evaluation ranking: 1 positive + EVAL_NEGATIVES_PER_QUERY negatives.
EVAL_NEGATIVES_PER_QUERY = 31

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Không giới hạn số lượng mẫu ở các tập
MAX_TRAIN_QUERIES = None
MAX_VALIDATION_QUERIES = None
MAX_TEST_QUERIES = None

print("Thiết bị:", DEVICE)
print("Tỷ lệ chia dữ liệu:", {"train": TRAIN_SIZE, "validation": VAL_SIZE, "test": TEST_SIZE})

In [ ]:
# Fix lỗi: RuntimeError: operator torchvision::nms does not exist
# Nguyên nhân thường là torchvision không tương thích với torch trong runtime.
# Với bài toán NLP embedding/reranking, ta không cần torchvision.
import os
import sys
import subprocess

os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    check=False,
)

print("Đã gỡ torchvision để tránh lỗi import transformers.")

## 4. Tải và chuẩn hóa dữ liệu


In [ ]:
def _clean_text(x) -> str:
    if x is None:
        return ""
    if isinstance(x, (list, tuple)):
        x = " ".join([str(i) for i in x])
    return str(x).strip()


def _parse_generated_qa_pairs(value):
    """Dataset thangvip thường là list[dict], nhưng hàm này cũng xử lý trường hợp bị lưu dạng string."""
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            return json.loads(value)
        except Exception:
            pass
        try:
            import ast
            return ast.literal_eval(value)
        except Exception:
            return []
    return []


def load_adamwhite625():
    print("Đang tải dataset: adamwhite625/vietnam-legal-qa...")
    try:
        ds = load_dataset("adamwhite625/vietnam-legal-qa", split="train")
        standardized_data = []
        for row in ds:
            q = _clean_text(row.get("question", ""))
            a = _clean_text(row.get("law_content", ""))
            if q and a and len(a) > 50:
                standardized_data.append({"query": q, "pos": a, "source": "adamwhite625/vietnam-legal-qa"})
        print(f"-> Thu thập được {len(standardized_data)} cặp.")
        return standardized_data
    except Exception as e:
        print(f"Lỗi khi tải adamwhite625/vietnam-legal-qa: {e}")
        return []


def load_huyydangg():
    print("Đang tải dataset: huyydangg/LEGAL-EVAL-Dataset...")
    try:
        ds = load_dataset("huyydangg/LEGAL-EVAL-Dataset", split="test")
        standardized_data = []
        for row in ds:
            q = _clean_text(row.get("anchor", ""))
            a = _clean_text(row.get("positive", ""))
            if q and a and len(a) > 50:
                standardized_data.append({"query": q, "pos": a, "source": "huyydangg/LEGAL-EVAL-Dataset"})
        print(f"-> Thu thập được {len(standardized_data)} cặp.")
        return standardized_data
    except Exception as e:
        print(f"Lỗi khi tải huyydangg/LEGAL-EVAL-Dataset: {e}")
        return []


def load_thangvip():
    print("Đang tải dataset: thangvip/vietnamese-legal-qa...")
    try:
        ds = load_dataset("thangvip/vietnamese-legal-qa", split="train")
        standardized_data = []
        for row in ds:
            qa_pairs = _parse_generated_qa_pairs(row.get("generated_qa_pairs", []))
            if not isinstance(qa_pairs, list):
                continue
            for pair in qa_pairs:
                if not isinstance(pair, dict):
                    continue
                q = _clean_text(pair.get("question", ""))
                a = _clean_text(pair.get("answer", ""))
                if q and a and len(a) > 50:
                    standardized_data.append({"query": q, "pos": a, "source": "thangvip/vietnamese-legal-qa"})
        print(f"-> Thu thập được {len(standardized_data)} cặp.")
        return standardized_data
    except Exception as e:
        print(f"Lỗi khi tải thangvip/vietnamese-legal-qa: {e}")
        return []


def deduplicate_pairs(data: List[Dict[str, str]]) -> List[Dict[str, str]]:
    seen = set()
    deduped = []
    for item in data:
        q = _clean_text(item.get("query"))
        p = _clean_text(item.get("pos"))
        key = (q, p)
        if q and p and key not in seen:
            seen.add(key)
            deduped.append({**item, "query": q, "pos": p})
    return deduped


In [ ]:
data1 = load_adamwhite625()
data2 = load_huyydangg()
data3 = load_thangvip()

all_data = deduplicate_pairs(data1 + data2 + data3)
random.shuffle(all_data)

n_total = len(all_data)
n_train = int(n_total * TRAIN_SIZE)
n_val = int(n_total * VAL_SIZE)
n_test = n_total - n_train - n_val

train_data = all_data[:n_train]
val_data = all_data[n_train:n_train + n_val]
test_data = all_data[n_train + n_val:]

if MAX_TRAIN_QUERIES is not None:
    train_data = train_data[:MAX_TRAIN_QUERIES]

if MAX_VAL_QUERIES is not None:
    val_data = val_data[:MAX_VAL_QUERIES]

if MAX_TEST_QUERIES is not None:
    test_data = test_data[:MAX_TEST_QUERIES]

eval_val_data = val_data[:MAX_VALIDATION_QUERIES] if MAX_VALIDATION_QUERIES is not None else val_data
eval_test_data = test_data[:MAX_TEST_QUERIES] if MAX_TEST_QUERIES is not None else test_data

print(f"Tổng số cặp Q&A hợp lệ sau deduplicate: {n_total}")
print(f"Số lượng Train: {len(train_data)}")
print(f"Số lượng Validation gốc: {len(val_data)}")
print(f"Số lượng Test gốc: {len(test_data)}")
print(f"Số lượng Validation dùng khi chạy nhanh: {len(eval_val_data)}")
print(f"Số lượng Test dùng khi chạy nhanh: {len(eval_test_data)}")

assert len(train_data) > 0 and len(val_data) > 0 and len(test_data) > 0, "Không có đủ dữ liệu để chia Train/Validation/Test."

split_summary = pd.DataFrame([
    {"Tập dữ liệu": "Train", "Số mẫu": len(train_data), "Tỷ lệ dự kiến": f"{TRAIN_SIZE:.0%}", "Vai trò": "Huấn luyện mô hình"},
    {"Tập dữ liệu": "Validation", "Số mẫu": len(val_data), "Tỷ lệ dự kiến": f"{VAL_SIZE:.0%}", "Vai trò": "Theo dõi learning curves và chọn checkpoint"},
    {"Tập dữ liệu": "Test", "Số mẫu": len(test_data), "Tỷ lệ dự kiến": f"{TEST_SIZE:.0%}", "Vai trò": "Đánh giá cuối cùng"},
])

print("Bảng 2. Thống kê số mẫu theo từng tập dữ liệu")
display(split_summary)

source_summary = pd.Series([x["source"] for x in all_data]).value_counts().reset_index()
source_summary.columns = ["Nguồn dữ liệu", "Số mẫu"]
print("Bảng 3. Phân bố mẫu theo nguồn dữ liệu")
display(source_summary)

### 4.1. Tiền xử lý dữ liệu

Các bước tiền xử lý đã áp dụng:

1. **Chuẩn hóa trường dữ liệu**: gom nhiều schema khác nhau về `query` và `pos`.
2. **Làm sạch chuỗi**: chuyển list/tuple sang chuỗi, loại bỏ khoảng trắng đầu/cuối.
3. **Lọc mẫu thiếu hoặc quá ngắn**: bỏ các mẫu không có câu hỏi/câu trả lời hoặc `pos` quá ngắn.
4. **Parse trường QA dạng JSON/string**: áp dụng cho dataset có trường `generated_qa_pairs`.
5. **Deduplicate**: loại bỏ cặp `(query, pos)` trùng nhau.
6. **Tokenization theo cặp**: tokenizer nhận đồng thời `(query, passage)`.
7. **Truncation**: nếu tổng độ dài `query + passage + special tokens` vượt `MAX_LENGTH`, phần dư sẽ bị cắt. Với dữ liệu pháp luật, phần bị cắt thường là đuôi của `passage` vì `query` thường ngắn hơn nhiều.
8. **Negative sampling**: sinh các đoạn không liên quan để mô hình học phân biệt positive với negatives.

Không dùng tăng cường dữ liệu ngôn ngữ vì trong bài toán pháp luật, việc paraphrase tự động có thể làm sai nghĩa pháp lý.


## 5. Lựa chọn mô hình và lý do

Notebook này chỉ thực hiện **một mô hình reranking**: `BAAI/bge-reranker-v2-m3`.

**Bảng 4. Mô hình được huấn luyện và đánh giá trong notebook**

| Mô hình | Loại | Vai trò | Lý do chọn |
|---|---|---|---|
| `BAAI/bge-reranker-v2-m3` | Cross-encoder reranker đa ngôn ngữ | Mô hình chính | Phù hợp với bài toán chấm điểm trực tiếp cặp `query` - `passage`, hỗ trợ nhiều ngôn ngữ và có khả năng xử lý văn bản dài hơn nhiều reranker thông thường. |

Trong bài toán reranking, cross-encoder thường chính xác hơn bi-encoder ở bước sắp xếp lại vì mô hình được nhìn thấy đồng thời cả `query` và `passage`, cho phép attention trực tiếp giữa hai văn bản.

Hai mô hình reranking khác sẽ được thực hiện trong notebook riêng của các thành viên khác, sau đó nhóm có thể ghép kết quả vào một bảng so sánh chung trong báo cáo cuối.


## 5.1. Tạo negatives cho reranker

Dữ liệu gốc chỉ có cặp positive `(query, pos)`. Để huấn luyện reranker, ta tạo group:

```text
query
 ├── positive passage
 ├── negative passage 1
 ├── negative passage 2
 └── negative passage 3
```

Notebook mặc định dùng `NEGATIVE_STRATEGY = "tfidf"`, tức là sinh **semi-hard negatives** bằng TF-IDF. Các negatives này thường gần query hơn random negatives theo từ vựng, nên khó hơn và hữu ích hơn cho reranker.

In [ ]:
def random_negatives_for_index(
    idx: int,
    data: List[Dict[str, str]],
    num_negatives: int,
    rng: random.Random,
) -> List[str]:
    pos = data[idx]["pos"]
    n = len(data)
    negatives = []
    attempts = 0
    while len(negatives) < num_negatives and attempts < 500:
        j = rng.randrange(n)
        candidate = data[j]["pos"]
        if j != idx and candidate != pos and candidate not in negatives:
            negatives.append(candidate)
        attempts += 1

    # Fallback tuyến tính nếu random chưa đủ.
    if len(negatives) < num_negatives:
        for j, item in enumerate(data):
            candidate = item["pos"]
            if j != idx and candidate != pos and candidate not in negatives:
                negatives.append(candidate)
                if len(negatives) == num_negatives:
                    break
    return negatives


def build_random_negatives(
    data: List[Dict[str, str]],
    num_negatives: int,
    seed: int = 42,
) -> List[List[str]]:
    rng = random.Random(seed)
    return [random_negatives_for_index(i, data, num_negatives, rng) for i in tqdm(range(len(data)), desc="Random negatives")]


def build_tfidf_negatives(
    data: List[Dict[str, str]],
    num_negatives: int,
    top_k_pool: int = 50,
    seed: int = 42,
    batch_size: int = 128,
) -> List[List[str]]:
    rng = random.Random(seed)
    queries = [x["query"] for x in data]
    passages = [x["pos"] for x in data]
    n = len(data)

    print("Đang fit TF-IDF trên corpus passages...")
    vectorizer = TfidfVectorizer(
        lowercase=True,
        max_features=100_000,
        ngram_range=(1, 2),
        min_df=2,
    )
    doc_matrix = vectorizer.fit_transform(passages)
    query_matrix = vectorizer.transform(queries)

    all_negatives = []
    k = min(top_k_pool + 1, n)

    for start in tqdm(range(0, n, batch_size), desc="TF-IDF negatives"):
        end = min(start + batch_size, n)
        sims = query_matrix[start:end].dot(doc_matrix.T).toarray()

        for local_i, scores in enumerate(sims):
            idx = start + local_i
            pos = passages[idx]

            # Lấy top-k ứng viên giống query nhất.
            if k < n:
                candidate_idx = np.argpartition(-scores, kth=k - 1)[:k]
            else:
                candidate_idx = np.arange(n)

            candidate_idx = candidate_idx[np.argsort(-scores[candidate_idx])]
            negatives = []
            for j in candidate_idx:
                candidate = passages[int(j)]
                if int(j) != idx and candidate != pos and candidate not in negatives:
                    negatives.append(candidate)
                if len(negatives) == num_negatives:
                    break

            if len(negatives) < num_negatives:
                extra = random_negatives_for_index(idx, data, num_negatives - len(negatives), rng)
                for item in extra:
                    if item not in negatives:
                        negatives.append(item)
                    if len(negatives) == num_negatives:
                        break

            all_negatives.append(negatives)

    return all_negatives


if NEGATIVE_STRATEGY == "tfidf":
    train_negatives = build_tfidf_negatives(
        train_data,
        num_negatives=TRAIN_NEGATIVES_PER_QUERY,
        top_k_pool=TFIDF_TOP_K_POOL,
        seed=SEED,
    )
elif NEGATIVE_STRATEGY == "random":
    train_negatives = build_random_negatives(
        train_data,
        num_negatives=TRAIN_NEGATIVES_PER_QUERY,
        seed=SEED,
    )
else:
    raise ValueError("NEGATIVE_STRATEGY phải là 'tfidf' hoặc 'random'.")

# Validation dùng random negatives để tính eval_loss/eval_top1_accuracy nhanh theo epoch.
val_group_negatives = build_random_negatives(
    eval_val_data,
    num_negatives=TRAIN_NEGATIVES_PER_QUERY,
    seed=SEED + 13,
)

print("Ví dụ training group:")
print("Query:", train_data[0]["query"][:300])
print("\nPositive:", train_data[0]["pos"][:300])
for i, neg in enumerate(train_negatives[0], 1):
    print(f"\nNegative {i}:", neg[:300])

## 6. Dataset, collator, kiến trúc và loss groupwise cho reranker

Mỗi sample huấn luyện là một group có `1 positive + K negatives`. Positive luôn được đặt ở vị trí 0, do đó nhãn đúng của mỗi group là class `0`.

Hàm mất mát:

```text
CrossEntropyLoss(scores_of_group, label=0)
```

Ý nghĩa: mô hình học để score của positive cao hơn các negative trong cùng group.

In [ ]:
class RerankerGroupDataset(Dataset):
    def __init__(self, data: List[Dict[str, str]], negatives: List[List[str]]):
        assert len(data) == len(negatives)
        self.data = data
        self.negatives = negatives

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        passages = [item["pos"]] + self.negatives[idx]
        return {
            "query": item["query"],
            "passages": passages,
        }


@dataclass
class RerankerGroupCollator:
    tokenizer: AutoTokenizer
    max_length: int

    def __call__(self, features):
        group_size = len(features[0]["passages"])
        queries = []
        passages = []

        for feature in features:
            assert len(feature["passages"]) == group_size, "Mọi group phải có cùng số passage."
            queries.extend([feature["query"]] * group_size)
            passages.extend(feature["passages"])

        batch = self.tokenizer(
            queries,
            passages,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        batch["group_size"] = torch.tensor(group_size, dtype=torch.long)
        # Label 0 vì positive passage luôn đứng đầu group.
        batch["labels"] = torch.zeros(len(features), dtype=torch.long)
        return batch


class GroupwiseRerankerTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        group_size = int(inputs.pop("group_size").item())
        labels = inputs.pop("labels", None)
        outputs = model(**inputs)
        logits = outputs.logits.view(-1, group_size)

        if labels is None:
            labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
        else:
            labels = labels.to(logits.device)

        loss = F.cross_entropy(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only=False, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        group_size = int(inputs.pop("group_size").item())
        labels = inputs.pop("labels", None)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits.view(-1, group_size)
            if labels is None:
                labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
            else:
                labels = labels.to(logits.device)
            loss = F.cross_entropy(logits, labels)

        if prediction_loss_only:
            return (loss.detach(), None, None)

        return (loss.detach(), logits.detach(), labels.detach())


def compute_group_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.asarray(logits)
    labels = np.asarray(labels)

    preds = np.argmax(logits, axis=1)
    top1_accuracy = float(np.mean(preds == labels))

    ranks = []
    for row, label in zip(logits, labels):
        sorted_idx = np.argsort(-row)
        rank = int(np.where(sorted_idx == label)[0][0]) + 1
        ranks.append(rank)

    mrr = float(np.mean([1.0 / r for r in ranks]))
    return {
        "top1_accuracy": top1_accuracy,
        "mrr": mrr,
    }


train_dataset = RerankerGroupDataset(train_data, train_negatives)
val_group_dataset = RerankerGroupDataset(eval_val_data, val_group_negatives)

print("Số training groups:", len(train_dataset))
print("Số validation groups dùng khi train:", len(val_group_dataset))
print("Group size:", 1 + TRAIN_NEGATIVES_PER_QUERY)

## 6.1. Load model và mô tả kiến trúc

Mô hình chính là một **cross-encoder reranker**. Input của mô hình là cặp `(query, passage)`, sau đó Transformer encoder mã hóa đồng thời cả hai đoạn và classification head trả về một logit relevance.

Các thông tin quan trọng phục vụ báo cáo:

- Số tham số tổng.
- Số tham số được fine-tune.
- Hàm kích hoạt trong config.
- Sơ đồ kiến trúc mức khái quát.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Tiết kiệm VRAM. Có thể tắt nếu gặp lỗi.
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

model.to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
hidden_act = getattr(model.config, "hidden_act", "Không có trong config")
num_hidden_layers = getattr(model.config, "num_hidden_layers", "Không có trong config")
hidden_size = getattr(model.config, "hidden_size", "Không có trong config")
num_attention_heads = getattr(model.config, "num_attention_heads", "Không có trong config")

arch_summary = pd.DataFrame([
    {"Thuộc tính": "Model", "Giá trị": MODEL_NAME},
    {"Thuộc tính": "Kiểu mô hình", "Giá trị": "Transformer encoder cross-encoder reranker"},
    {"Thuộc tính": "Số hidden layers", "Giá trị": num_hidden_layers},
    {"Thuộc tính": "Hidden size", "Giá trị": hidden_size},
    {"Thuộc tính": "Số attention heads", "Giá trị": num_attention_heads},
    {"Thuộc tính": "Activation trong config", "Giá trị": hidden_act},
    {"Thuộc tính": "Tổng số tham số", "Giá trị": f"{num_params:,}"},
    {"Thuộc tính": "Số tham số trainable", "Giá trị": f"{num_trainable:,}"},
])
print("Bảng 5. Thông tin kiến trúc và số tham số của mô hình")
display(arch_summary)

In [ ]:
def draw_architecture_diagram():
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.axis("off")

    boxes = [
        (0.05, 0.55, "Query"),
        (0.05, 0.15, "Passage"),
        (0.25, 0.35, "Tokenizer\n[CLS] query [SEP] passage [SEP]"),
        (0.52, 0.35, "Transformer Encoder\nSelf-Attention + FFN"),
        (0.78, 0.35, "Classification Head\nLinear → relevance logit"),
    ]

    for x, y, text in boxes:
        rect = plt.Rectangle((x, y), 0.18, 0.22, fill=False)
        ax.add_patch(rect)
        ax.text(x + 0.09, y + 0.11, text, ha="center", va="center", fontsize=9)

    arrows = [
        ((0.23, 0.66), (0.25, 0.46)),
        ((0.23, 0.26), (0.25, 0.46)),
        ((0.43, 0.46), (0.52, 0.46)),
        ((0.70, 0.46), (0.78, 0.46)),
    ]

    for start, end in arrows:
        ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->"))

    ax.set_title("Sơ đồ kiến trúc reranker dạng cross-encoder")
    plt.show()

draw_architecture_diagram()


*Hình 1. Sơ đồ kiến trúc tổng quát của reranker cross-encoder. Mô hình nhận đồng thời query và passage, sau đó sinh một relevance logit dùng để xếp hạng.*


## 7. Cấu hình huấn luyện và siêu tham số

**Bảng 6. Cấu hình huấn luyện**

| Thành phần | Giá trị | Giải thích |
|---|---|---|
| Loss | Groupwise Cross-Entropy | Positive phải có score cao nhất trong group |
| Optimizer | AdamW mặc định của Hugging Face Trainer | Phù hợp fine-tune Transformer |
| Learning rate | `2e-5` | Giá trị phổ biến khi fine-tune Transformer |
| Scheduler | Linear scheduler với warmup | Dùng `warmup_ratio=0.10` |
| Weight decay | `0.01` | Regularization L2 |
| Epochs | `2` | Giảm rủi ro overfitting và tiết kiệm thời gian |
| Batch size | `2 query-groups/device` | Do cross-encoder tốn VRAM |
| Gradient accumulation | `8` | Tăng effective batch size |
| Max length | `1024 tokens` | Cân bằng giữa ngữ cảnh pháp luật và VRAM |
| Negative strategy | `TF-IDF semi-hard negatives` | Khó hơn random negatives |

Phương pháp tinh chỉnh siêu tham số trong notebook là **thử nghiệm thủ công có kiểm soát**. Có thể mở rộng sang grid search/random search nếu có nhiều GPU hơn.


In [ ]:
hyperparam_table = pd.DataFrame([
    {"Siêu tham số": "MODEL_NAME", "Giá trị": MODEL_NAME},
    {"Siêu tham số": "MAX_LENGTH", "Giá trị": MAX_LENGTH},
    {"Siêu tham số": "TRAIN_NEGATIVES_PER_QUERY", "Giá trị": TRAIN_NEGATIVES_PER_QUERY},
    {"Siêu tham số": "PER_DEVICE_TRAIN_BATCH_SIZE", "Giá trị": PER_DEVICE_TRAIN_BATCH_SIZE},
    {"Siêu tham số": "GRADIENT_ACCUMULATION_STEPS", "Giá trị": GRADIENT_ACCUMULATION_STEPS},
    {"Siêu tham số": "EPOCHS", "Giá trị": EPOCHS},
    {"Siêu tham số": "LEARNING_RATE", "Giá trị": LEARNING_RATE},
    {"Siêu tham số": "WARMUP_RATIO", "Giá trị": WARMUP_RATIO},
    {"Siêu tham số": "WEIGHT_DECAY", "Giá trị": WEIGHT_DECAY},
    {"Siêu tham số": "NEGATIVE_STRATEGY", "Giá trị": NEGATIVE_STRATEGY},
    {"Siêu tham số": "TFIDF_TOP_K_POOL", "Giá trị": TFIDF_TOP_K_POOL},
])
print("Bảng 7. Danh sách siêu tham số chính")
display(hyperparam_table)


## 8. Hàm đánh giá ranking và classification view

Các metric ranking chính:

- `MRR@10`: positive càng đứng cao thì càng tốt.
- `NDCG@10`: chất lượng ranking trong top 10.
- `Recall@k`: tỷ lệ query có positive nằm trong top-k.
- `MeanRank`, `MedianRank`: thứ hạng trung bình/trung vị của positive.

Để đáp ứng yêu cầu báo cáo về `Accuracy/Precision/Recall/F1` và `Confusion Matrix`, notebook bổ sung **top-1 classification view**:

- Với mỗi query, passage được model xếp top 1 được xem là dự đoán `relevant`.
- Passage positive thật là nhãn `relevant`.
- Các passage còn lại là `non-relevant`.

Cách nhìn này phù hợp cho reranking, nhưng cần lưu ý `Accuracy` có thể cao do số negative nhiều.


In [ ]:
@torch.no_grad()
def score_pairs(
    model,
    tokenizer,
    pairs: List[Tuple[str, str]],
    batch_size: int = 16,
    max_length: int = 1024,
    normalize: bool = False,
) -> List[float]:
    model.eval()
    scores = []

    for start in range(0, len(pairs), batch_size):
        batch_pairs = pairs[start:start + batch_size]
        queries = [x[0] for x in batch_pairs]
        passages = [x[1] for x in batch_pairs]

        encoded = tokenizer(
            queries,
            passages,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(model.device)

        logits = model(**encoded).logits.view(-1)
        batch_scores = logits.detach().float().cpu().numpy()

        if normalize:
            batch_scores = 1 / (1 + np.exp(-batch_scores))

        scores.extend(batch_scores.tolist())

    return scores


def build_ranking_candidates(
    eval_data: List[Dict[str, str]],
    idx: int,
    num_negatives: int,
    rng: random.Random,
) -> Tuple[str, List[str], int]:
    item = eval_data[idx]
    query = item["query"]
    positive = item["pos"]

    negatives = []
    attempts = 0
    while len(negatives) < num_negatives and attempts < 1000:
        j = rng.randrange(len(eval_data))
        candidate = eval_data[j]["pos"]
        if j != idx and candidate != positive and candidate not in negatives:
            negatives.append(candidate)
        attempts += 1

    # Fallback tuyến tính nếu eval_data nhỏ.
    if len(negatives) < num_negatives:
        for j, other in enumerate(eval_data):
            candidate = other["pos"]
            if j != idx and candidate != positive and candidate not in negatives:
                negatives.append(candidate)
                if len(negatives) == num_negatives:
                    break

    candidates = [positive] + negatives
    order = list(range(len(candidates)))
    rng.shuffle(order)

    shuffled_candidates = [candidates[i] for i in order]
    positive_new_idx = order.index(0)
    return query, shuffled_candidates, positive_new_idx


def compute_ranking_metrics(ranks: List[int], candidates_per_query: int) -> Dict[str, float]:
    ranks = np.array(ranks)
    metrics = {
        "MRR@10": float(np.mean([1.0 / r if r <= 10 else 0.0 for r in ranks])),
        "NDCG@10": float(np.mean([1.0 / math.log2(r + 1) if r <= 10 else 0.0 for r in ranks])),
        "Recall@1": float(np.mean(ranks <= 1)),
        "Recall@3": float(np.mean(ranks <= 3)),
        "Recall@5": float(np.mean(ranks <= 5)),
        "Recall@10": float(np.mean(ranks <= 10)),
        "MeanRank": float(np.mean(ranks)),
        "MedianRank": float(np.median(ranks)),
        "NumQueries": int(len(ranks)),
        "CandidatesPerQuery": int(candidates_per_query),
    }
    return metrics


def top1_classification_metrics_from_ranks(
    ranks: List[int],
    candidates_per_query: int,
) -> Tuple[Dict[str, float], np.ndarray]:
    ranks = np.array(ranks)
    n = len(ranks)
    tp = int(np.sum(ranks == 1))
    fn = int(n - tp)
    fp = int(n - tp)
    tn = int(n * (candidates_per_query - 1) - fp)

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    metrics = {
        "Top1_Accuracy": float(accuracy),
        "Top1_Precision": float(precision),
        "Top1_Recall": float(recall),
        "Top1_F1": float(f1),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }
    cm = np.array([[tn, fp], [fn, tp]])
    return metrics, cm


def evaluate_reranker(
    model,
    tokenizer,
    eval_data: List[Dict[str, str]],
    num_negatives: int = 31,
    batch_size: int = 16,
    max_length: int = 1024,
    seed: int = 42,
    return_details: bool = False,
):
    rng = random.Random(seed)
    ranks = []
    details = []

    for idx in tqdm(range(len(eval_data)), desc="Evaluating reranker"):
        query, candidates, positive_idx = build_ranking_candidates(eval_data, idx, num_negatives, rng)
        pairs = [(query, passage) for passage in candidates]
        scores = score_pairs(
            model,
            tokenizer,
            pairs,
            batch_size=batch_size,
            max_length=max_length,
            normalize=False,
        )

        sorted_idx = np.argsort(-np.array(scores))
        rank = int(np.where(sorted_idx == positive_idx)[0][0]) + 1
        ranks.append(rank)

        if return_details:
            top_idx = int(sorted_idx[0])
            details.append({
                "query": query,
                "positive_passage": candidates[positive_idx],
                "predicted_top_passage": candidates[top_idx],
                "positive_rank": rank,
                "positive_score": float(scores[positive_idx]),
                "top_score": float(scores[top_idx]),
                "candidates_per_query": len(candidates),
            })

    metrics = compute_ranking_metrics(ranks, candidates_per_query=1 + num_negatives)
    cls_metrics, cm = top1_classification_metrics_from_ranks(ranks, candidates_per_query=1 + num_negatives)
    metrics.update(cls_metrics)

    if return_details:
        return metrics, ranks, cm, pd.DataFrame(details)
    return metrics


def print_metrics(metrics: Dict[str, float]):
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

## 9. Fine-tuning

Trainer được cấu hình để đánh giá trên validation set sau mỗi epoch. Điều này giúp vẽ learning curves và phân tích mô hình có hội tụ/overfit hay không.

In [ ]:
training_args_kwargs = dict(
    output_dir=OUTPUT_PATH,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_top1_accuracy",
    greater_is_better=True,
)

# Tương thích nhiều phiên bản transformers: evaluation_strategy cũ và eval_strategy mới.
ta_params = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in ta_params:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

trainer = GroupwiseRerankerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_group_dataset,
    data_collator=RerankerGroupCollator(tokenizer=tokenizer, max_length=MAX_LENGTH),
    compute_metrics=compute_group_metrics,
)

print("=" * 60)
print("BẮT ĐẦU FINE-TUNING RERANKER")
print("=" * 60)

trainer.train()

trainer.save_model(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)

print(f"\n✅ Hoàn tất fine-tune. Model đã lưu tại: {OUTPUT_PATH}")

### 9.1. Learning curves: Loss và metric trên Train/Validation

Các biểu đồ dưới đây phục vụ phần **Kết quả thực nghiệm** của báo cáo.  
Train loss lấy từ log huấn luyện, validation loss và validation top-1 accuracy được tính sau mỗi epoch.


In [ ]:
history = pd.DataFrame(trainer.state.log_history)
history.to_csv("./training_log_history.csv", index=False, encoding="utf-8-sig")

print("Bảng 8. Log huấn luyện từ Trainer")
display(history)

# Learning curve: train loss và validation loss
plt.figure(figsize=(8, 5))

train_logs = history.dropna(subset=["loss"]) if "loss" in history.columns else pd.DataFrame()
eval_logs = history.dropna(subset=["eval_loss"]) if "eval_loss" in history.columns else pd.DataFrame()

if not train_logs.empty:
    x_train = train_logs["epoch"] if "epoch" in train_logs.columns else range(len(train_logs))
    plt.plot(x_train, train_logs["loss"], marker="o", label="Train loss")

if not eval_logs.empty:
    x_eval = eval_logs["epoch"] if "epoch" in eval_logs.columns else range(len(eval_logs))
    plt.plot(x_eval, eval_logs["eval_loss"], marker="o", label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curve: Train Loss và Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# Metric curve: validation top-1 accuracy và MRR
plt.figure(figsize=(8, 5))

if "eval_top1_accuracy" in history.columns:
    metric_logs = history.dropna(subset=["eval_top1_accuracy"])
    x_metric = metric_logs["epoch"] if "epoch" in metric_logs.columns else range(len(metric_logs))
    plt.plot(x_metric, metric_logs["eval_top1_accuracy"], marker="o", label="Validation Top-1 Accuracy")

if "eval_mrr" in history.columns:
    metric_logs = history.dropna(subset=["eval_mrr"])
    x_metric = metric_logs["epoch"] if "epoch" in metric_logs.columns else range(len(metric_logs))
    plt.plot(x_metric, metric_logs["eval_mrr"], marker="o", label="Validation MRR")

plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.title("Learning Curve: Validation Ranking Metrics")
plt.legend()
plt.grid(True)
plt.show()

*Hình 2. Learning curve của train loss và validation loss theo epoch. Biểu đồ dùng để nhận xét mô hình có hội tụ hay không và có dấu hiệu overfitting hay không.*

*Hình 3. Learning curve của validation top-1 accuracy/MRR theo epoch. Metric tăng ổn định thường cho thấy mô hình học được tín hiệu ranking từ dữ liệu.*


## 10. Đánh giá trên tập Test sau fine-tune

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 60)
print("ĐÁNH GIÁ MODEL SAU FINE-TUNE TRÊN TEST SET")
print("=" * 60)

finetuned_metrics, finetuned_ranks, finetuned_cm, finetuned_details = evaluate_reranker(
    model,
    tokenizer,
    eval_data=eval_test_data,
    num_negatives=EVAL_NEGATIVES_PER_QUERY,
    batch_size=PER_DEVICE_TRAIN_BATCH_SIZE * (1 + TRAIN_NEGATIVES_PER_QUERY),
    max_length=MAX_LENGTH,
    seed=SEED,
    return_details=True,
)

print_metrics(finetuned_metrics)

os.makedirs("./eval_reranker_finetuned", exist_ok=True)
with open("./eval_reranker_finetuned/metrics.json", "w", encoding="utf-8") as f:
    json.dump(finetuned_metrics, f, ensure_ascii=False, indent=2)

finetuned_details.to_csv("./eval_reranker_finetuned/error_details.csv", index=False, encoding="utf-8-sig")

### 10.1. Bảng metric định lượng trên Test

Bảng này là kết quả của **một mô hình duy nhất** sau fine-tune. Kết quả so sánh với hai mô hình reranking khác sẽ được tổng hợp ở báo cáo nhóm.


In [ ]:
result_summary = pd.DataFrame([
    finetuned_metrics,
], index=[
    "BGE reranker v2-m3 fine-tuned",
])

important_cols = [
    "MRR@10", "NDCG@10", "Recall@1", "Recall@3", "Recall@5", "Recall@10",
    "Top1_Accuracy", "Top1_Precision", "Top1_Recall", "Top1_F1",
    "MeanRank", "MedianRank", "NumQueries", "CandidatesPerQuery",
]
result_summary = result_summary[[c for c in important_cols if c in result_summary.columns]]

print("Bảng 9. Kết quả định lượng của mô hình BGE reranker v2-m3 fine-tuned trên tập Test")
display(result_summary)

result_summary.to_csv("./eval_reranker_finetuned/test_metrics_summary.csv", encoding="utf-8-sig")


### 10.2. Confusion Matrix theo top-1 classification view


In [ ]:
print("Bảng 10. Confusion Matrix của mô hình fine-tuned trên Test")
cm_df = pd.DataFrame(
    finetuned_cm,
    index=["True non-relevant", "True relevant"],
    columns=["Pred non-relevant", "Pred relevant"],
)
display(cm_df)

disp = ConfusionMatrixDisplay(
    confusion_matrix=finetuned_cm,
    display_labels=["Non-relevant", "Relevant"],
)
disp.plot(values_format="d")
plt.title("Confusion Matrix - Fine-tuned Reranker")
plt.show()


*Hình 4. Confusion Matrix của mô hình fine-tuned trên test set theo top-1 classification view. Một query được xem là đúng nếu passage positive được xếp ở vị trí top 1.*


### 10.3. Biểu đồ metric chính của mô hình


In [ ]:
plot_cols = [c for c in ["MRR@10", "NDCG@10", "Recall@1", "Recall@5", "Recall@10", "Top1_F1"] if c in result_summary.columns]

metric_values = result_summary.loc["BGE reranker v2-m3 fine-tuned", plot_cols]
plt.figure(figsize=(9, 4))
metric_values.plot(kind="bar")
plt.ylabel("Giá trị metric")
plt.title("Các metric chính của mô hình BGE reranker v2-m3 fine-tuned")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 1.05)
plt.grid(axis="y")
plt.show()


*Hình 5. Biểu đồ các metric chính của mô hình `BAAI/bge-reranker-v2-m3` sau fine-tune trên tập test.*


## 11. Thảo luận: Overfitting/Underfitting và mức độ hội tụ

Cell dưới đây tự động đưa ra nhận xét sơ bộ dựa trên log huấn luyện. Khi viết báo cáo, cần kết hợp với quan sát trực tiếp learning curves.


In [ ]:
def analyze_training_behavior(history_df: pd.DataFrame):
    comments = []

    train_logs = history_df.dropna(subset=["loss"]) if "loss" in history_df.columns else pd.DataFrame()
    eval_logs = history_df.dropna(subset=["eval_loss"]) if "eval_loss" in history_df.columns else pd.DataFrame()

    if len(train_logs) >= 2:
        first_train = float(train_logs["loss"].iloc[0])
        last_train = float(train_logs["loss"].iloc[-1])
        if last_train < first_train:
            comments.append("Train loss giảm so với ban đầu, cho thấy mô hình có học được tín hiệu từ dữ liệu.")
        else:
            comments.append("Train loss chưa giảm rõ ràng; có thể cần tăng epoch, kiểm tra learning rate hoặc chất lượng dữ liệu.")

    if len(eval_logs) >= 2:
        first_eval = float(eval_logs["eval_loss"].iloc[0])
        last_eval = float(eval_logs["eval_loss"].iloc[-1])
        if last_eval <= first_eval:
            comments.append("Validation loss không tăng so với ban đầu, chưa thấy dấu hiệu overfitting rõ rệt.")
        else:
            comments.append("Validation loss tăng, trong khi train loss có thể giảm; đây là dấu hiệu cần kiểm tra overfitting.")

    if "eval_top1_accuracy" in history_df.columns:
        metric_logs = history_df.dropna(subset=["eval_top1_accuracy"])
        if len(metric_logs) >= 2:
            first_acc = float(metric_logs["eval_top1_accuracy"].iloc[0])
            last_acc = float(metric_logs["eval_top1_accuracy"].iloc[-1])
            if last_acc >= first_acc:
                comments.append("Validation top-1 accuracy tăng hoặc giữ ổn định, cho thấy chất lượng ranking cải thiện.")
            else:
                comments.append("Validation top-1 accuracy giảm; cần kiểm tra overfitting hoặc negative sampling quá nhiễu.")

    if not comments:
        comments.append("Chưa đủ log để kết luận. Cần chạy training ít nhất 2 epoch hoặc tăng logging/evaluation.")

    return comments

print("Nhận xét sơ bộ về quá trình học:")
for c in analyze_training_behavior(history):
    print("-", c)


Một số hướng xử lý nếu có overfitting:

- Giảm số epoch.
- Tăng weight decay.
- Dùng early stopping.
- Tăng chất lượng negatives thay vì chỉ tăng số lượng.
- Chunk passage dài để tránh mất thông tin quan trọng do truncation.
- Dùng validation set cố định và không điều chỉnh theo test set.


## 12. Phân tích lỗi

Phần này lấy các trường hợp mà positive passage không được xếp top 1. Các mẫu này nên được đưa vào báo cáo để giải thích nguyên nhân sai.


In [ ]:
def error_table_from_details(details_df: pd.DataFrame, max_rows: int = 20) -> pd.DataFrame:
    if details_df.empty:
        return details_df

    errors = details_df[details_df["positive_rank"] > 1].copy()
    if errors.empty:
        return errors

    errors["query_short"] = errors["query"].str.slice(0, 250)
    errors["positive_short"] = errors["positive_passage"].str.slice(0, 350)
    errors["predicted_top_short"] = errors["predicted_top_passage"].str.slice(0, 350)
    errors["score_gap_top_minus_positive"] = errors["top_score"] - errors["positive_score"]

    cols = [
        "positive_rank",
        "score_gap_top_minus_positive",
        "query_short",
        "positive_short",
        "predicted_top_short",
    ]
    return errors[cols].sort_values(["positive_rank", "score_gap_top_minus_positive"], ascending=[False, False]).head(max_rows)

error_examples = error_table_from_details(finetuned_details, max_rows=20)

print("Bảng 11. Một số trường hợp mô hình fine-tuned xếp sai trên Test")
display(error_examples)

error_examples.to_csv("./eval_reranker_finetuned/error_examples_for_report.csv", index=False, encoding="utf-8-sig")


Gợi ý phân tích lỗi trong báo cáo:

- **Nhiễu nhãn**: positive passage trong dataset có thể chưa phải đoạn phù hợp nhất.
- **Nhiều đoạn cùng đúng**: negative lấy từ dataset khác có thể cũng liên quan về mặt pháp lý.
- **Passage quá dài bị cắt**: thông tin quan trọng có thể nằm sau vị trí `MAX_LENGTH`.
- **Khớp từ vựng gây nhiễu**: đoạn sai có nhiều thuật ngữ giống query nên được score cao.
- **Thiếu ngữ cảnh pháp lý**: một số câu hỏi cần điều luật, khoản, nghị định cụ thể mà passage không nêu đầy đủ.


## 13. Test inference thủ công

In [ ]:
sample_query = "Người lao động nghỉ việc có được hưởng trợ cấp thôi việc không?"
sample_passage_good = "Người lao động đã làm việc thường xuyên từ đủ 12 tháng trở lên khi chấm dứt hợp đồng lao động đúng quy định có thể được người sử dụng lao động chi trả trợ cấp thôi việc, trừ các trường hợp pháp luật loại trừ."
sample_passage_bad = "Doanh nghiệp phải kê khai thuế giá trị gia tăng theo tháng hoặc theo quý tùy thuộc điều kiện pháp luật về thuế."

pairs = [
    (sample_query, sample_passage_good),
    (sample_query, sample_passage_bad),
]

raw_scores = score_pairs(model, tokenizer, pairs, batch_size=2, max_length=MAX_LENGTH, normalize=False)
norm_scores = score_pairs(model, tokenizer, pairs, batch_size=2, max_length=MAX_LENGTH, normalize=True)

for pair, raw, norm in zip(pairs, raw_scores, norm_scores):
    print("=" * 80)
    print("Query:", pair[0])
    print("Passage:", pair[1])
    print(f"Raw score: {raw:.4f}")
    print(f"Sigmoid score: {norm:.4f}")


## 14. Lưu model, kết quả và nén artifact

In [ ]:
# Lưu lại toàn bộ kết quả cần cho báo cáo.
os.makedirs("./report_artifacts", exist_ok=True)

split_summary.to_csv("./report_artifacts/table_2_split_summary.csv", index=False, encoding="utf-8-sig")
source_summary.to_csv("./report_artifacts/table_3_source_summary.csv", index=False, encoding="utf-8-sig")
arch_summary.to_csv("./report_artifacts/table_5_arch_summary.csv", index=False, encoding="utf-8-sig")
hyperparam_table.to_csv("./report_artifacts/table_7_hyperparams.csv", index=False, encoding="utf-8-sig")
result_summary.to_csv("./report_artifacts/table_9_test_metrics_single_model.csv", encoding="utf-8-sig")
cm_df.to_csv("./report_artifacts/table_10_confusion_matrix.csv", encoding="utf-8-sig")
error_examples.to_csv("./report_artifacts/table_11_error_examples.csv", index=False, encoding="utf-8-sig")

print("Đã lưu các bảng vào thư mục ./report_artifacts")


In [ ]:
!zip -r vietlaw-bge-reranker-v2-m3-finetuned.zip vietlaw-bge-reranker-v2-m3-finetuned/ report_artifacts/ training_log_history.csv

## 15. Kết luận mẫu cho báo cáo

Sau khi chạy notebook, phần kết luận nên dựa trên số liệu thực tế ở `Bảng 9`, `Hình 2`, `Hình 3`, `Hình 5` và `Bảng 11`.

Mẫu kết luận:

> Kết quả thực nghiệm cho thấy mô hình `BAAI/bge-reranker-v2-m3` sau fine-tune có khả năng xếp hạng các đoạn văn bản pháp luật liên quan cho mỗi câu hỏi dựa trên các metric như MRR@10, Recall@1, Recall@10 và F1 theo top-1 classification view. Learning curves trên tập train/validation được sử dụng để đánh giá mức độ hội tụ và phát hiện overfitting/underfitting. Một số lỗi có thể xuất hiện khi negative passage có thuật ngữ pháp lý gần với query, khi positive passage quá dài bị cắt bớt do `MAX_LENGTH`, hoặc khi có nhiều passage cùng đúng về mặt ngữ nghĩa. Trong báo cáo nhóm, kết quả của mô hình này sẽ được so sánh với hai mô hình reranking khác do các thành viên còn lại thực hiện.

## 16. Tài liệu tham khảo và nguồn

1. BAAI. `BAAI/bge-reranker-v2-m3`, Hugging Face model card.  
2. Xiao et al. BGE-M3: Multi-Lingual, Multi-Functionality, Multi-Granularity Text Embeddings.  
3. Vaswani et al. *Attention Is All You Need*.  
4. Hugging Face Transformers Documentation.  
5. Hugging Face Datasets Documentation.
